In [1]:
pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------  2.1/2.1 MB 23.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 4.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.model_selection import cross_val_score, StratifiedKFold
import optuna
from sklearn.metrics import mean_squared_error

c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df = pd.read_parquet(r"C:\Users\ADMIN\Downloads\train_val.parquet")
df.head()

,id,item_id,store_id,state_id,dept_id,cat_id,d,demand,lag_t7,rolling_mean_lag7_w7,...,year,event_name_1,event_name_2,snap_CA,snap_TX,snap_WI,sell_price,disaster_type,disaster_severity_flag,flag_ufc
0,HOBBIES_1_001_CA_1,1438,1,1,4,2,1029,0.0,0.0,1.000000,...,2013,1,1,0,0,0,8.26,None,0,0
1,HOBBIES_1_002_CA_1,1439,1,1,4,2,1029,0.0,1.0,0.142857,...,2013,1,1,0,0,0,3.97,None,0,0
2,HOBBIES_1_003_CA_1,1440,1,1,4,2,1029,0.0,0.0,0.000000,...,2013,1,1,0,0,0,NaN,None,0,0
3,HOBBIES_1_004_CA_1,1441,1,1,4,2,1029,3.0,0.0,1.142857,...,2013,1,1,0,0,0,4.64,None,0,0
4,HOBBIES_1_005_CA_1,1442,1,1,4,2,1029,1.0,1.0,0.428571,...,2013,1,1,0,0,0,3.08,None,0,0


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import os
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.model_selection import cross_val_score, StratifiedKFold
import optuna
from sklearn.metrics import mean_squared_error # import mean_squared_error

LAGS = [7, 28]
WINDOWS = [7, 28]
FIRST = 1942 # first to predict
LENGTH = 28

def demand_features_eval(df):
    """ Same as demand_features but for the step-by-step evaluation """
    out = df.groupby('id', sort=False).last()
    for lag in LAGS:
        out[f'lag_t{lag}'] = df.groupby('id', sort=False)['demand'].nth(-lag-1).astype("float32")
        for w in WINDOWS:
            out[f'rolling_mean_lag{lag}_w{w}'] = df.groupby('id', sort=False)['demand'].nth(list(range(-lag-w, -lag))).groupby('id', sort=False).mean().astype("float32")

    return out.reset_index()

def prep_train(df,drop_d = 1000):

    # Convert 'date' to numerical features
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek

    # One-hot encode 'disaster_type'
    df = pd.get_dummies(df, columns=['disaster_type'], prefix=['disaster_type'])

    x = list(set(df.columns) - {'id', 'd', 'demand','date'}) # exclude 'date' from features
    print(x)
    # Split into test, valid, train
    test = df[df.d >= FIRST - max(LAGS) - max(WINDOWS) - 28]
    df = df[df.d < FIRST]

    xtrain, xvalid, ytrain, yvalid = train_test_split(df[x], df["demand"], test_size=0.1, shuffle=True, random_state=54)
    train = lgb.Dataset(xtrain, label = ytrain)
    valid = lgb.Dataset(xvalid, label = yvalid)

    # return validation data for Optuna
    return train, valid, test, x, xvalid, yvalid

train, valid, test, x, X_valid, y_valid = prep_train(df, 1000 - 28)

def objective(trial):
    # Gợi ý các hyperparameter
    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'feature_pre_filter': False,
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 256),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }

    # Train
    gbm = lgb.train(
        param,
        train,
        valid_sets=[valid],
        num_boost_round=1000,
    )

    # Predict & Evaluate
    preds = gbm.predict(X_valid)
    rmse = mean_squared_error(y_valid, preds, squared=False)  # RMSE

    return rmse  # Tối ưu RMSE → minimize

# Study
study = optuna.create_study(direction='minimize')  # Vì RMSE càng thấp càng tốt
study.optimize(objective, n_trials=50)

# Kết quả
print("Best trial:")
trial = study.best_trial
print(f"  RMSE: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

['snap_CA', 'dayofweek', 'disaster_type_Severe Storm(s)', 'disaster_type_Flood', 'year', 'dept_id', 'store_id', 'lag_t7', 'rolling_mean_lag7_w28', 'cat_id', 'lag_t28', 'sell_price', 'state_id', 'item_id', 'disaster_type_Fire', 'rolling_mean_lag7_w7', 'snap_WI', 'flag_ufc', 'disaster_type_Earthquake', 'rolling_mean_lag28_w28', 'event_name_1', 'event_name_2', 'wday', 'snap_TX', 'disaster_severity_flag', 'day', 'month', 'rolling_mean_lag28_w7']


[I 2025-05-03 15:30:00,859] A new study created in memory with name: no-name-cee600a7-aa66-41fb-af13-8bfd81574d85
c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
[I 2025-05-03 15:49:22,703] Trial 0 finished with value: 2.1287191565302996 and parameters: {'lambda_l1': 9.766880213641771, 'lambda_l2': 0.009409688150802413, 'num_leaves': 121, 'feature_fraction': 0.6771942477442274, 'bagging_fraction': 0.5186056221239016, 'bagging_freq': 2, 'min_child_samples': 65}. Best is trial 0 with value: 2.1287191565302996.
c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the 

KeyboardInterrupt: 